In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU is available!")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not available. Using CPU.")

In [ ]:
!pip install -q transformers torch accelerate
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
print("Installation Complete")
model_id = "Qwen/Qwen2.5-0.5B-Instruct" # A tiny powerhouse (0.5B parameters)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="auto")
print("Model Downloaded")

In [ ]:
# 1. Print Architecture
print(model)

# 2. Total Layer Count
layer_count = len(model.model.layers)
print(f"\nTotal Layers: {layer_count}")

# 3. Identify where layers live
# For Qwen, they are in: model.model.layers
print(f"The first layer module: {model.model.layers[0]}")

In [ ]:
print("Model type:", type(model))
print("Backbone type:", type(model.model))
print("Embedding layer:\n", model.model.embed_tokens)
print("\nFinal norm:\n", model.model.norm)
print("\nLM head:\n", model.lm_head)

print("\nNumber of transformer layers:", len(model.model.layers))

for i in [0, 1, 12, 23]:
    print(f"\n--- Layer {i} ---")
    print(model.model.layers[i])

In [ ]:
def count_params(module):
    return sum(p.numel() for p in module.parameters())

print("Embedding params:", count_params(model.model.embed_tokens))
print("All transformer layers params:", count_params(model.model.layers))
print("Final norm params:", count_params(model.model.norm))
print("LM head params:", count_params(model.lm_head))
print("Total params:", count_params(model))

In [ ]:
def generate_text(prompt, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_prompts = [
    "What is the capital of Japan?",
    "Solve: 17 + 28 = ?",
    "Write a Python function to add two numbers.",
    "Explain recursion in one simple paragraph."
]

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{'='*60}")
    print(f"Prompt {i}: {prompt}")
    print(f"{'-'*60}")
    print(generate_text(prompt))

In [ ]:
import copy
from torch import nn

def load_fresh_model(model_id="Qwen/Qwen2.5-0.5B-Instruct"):
    fresh_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype="auto",
        device_map="auto"
    )
    return fresh_model

def prune_layers_keep_indices(model, keep_indices):
    keep_indices = sorted(keep_indices)
    new_layers = nn.ModuleList([model.model.layers[i] for i in keep_indices])
    model.model.layers = new_layers

    if hasattr(model.config, "num_hidden_layers"):
        model.config.num_hidden_layers = len(keep_indices)

    return model

def describe_layers(model):
    print("Remaining layers:", len(model.model.layers))
    print("Layer indices now run from 0 to", len(model.model.layers) - 1)

In [ ]:
base_layer_count = 24
keep_last2_removed = list(range(base_layer_count - 2))   # keep 0..21

pruned_model_last2 = load_fresh_model(model_id)
pruned_model_last2 = prune_layers_keep_indices(pruned_model_last2, keep_last2_removed)
describe_layers(pruned_model_last2)

In [ ]:
keep_first2_removed = list(range(2, base_layer_count))   # keep 2..23

pruned_model_first2 = load_fresh_model(model_id)
pruned_model_first2 = prune_layers_keep_indices(pruned_model_first2, keep_first2_removed)
describe_layers(pruned_model_first2)

In [ ]:
middle_remove = [11, 12]
keep_middle_removed = [i for i in range(base_layer_count) if i not in middle_remove]

pruned_model_middle2 = load_fresh_model(model_id)
pruned_model_middle2 = prune_layers_keep_indices(pruned_model_middle2, keep_middle_removed)
describe_layers(pruned_model_middle2)

In [ ]:
def generate_text_with_model(model_obj, prompt, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors="pt").to(model_obj.device)
    with torch.no_grad():
        outputs = model_obj.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
for prompt in test_prompts:
    print("\n" + "="*60)
    print("PROMPT:", prompt)
    print("-"*60)
    print("BASE MODEL:")
    print(generate_text_with_model(model, prompt))
    print("\nPRUNED (last 2 removed):")
    print(generate_text_with_model(pruned_model_last2, prompt))

In [ ]:
for prompt in test_prompts:
    print("\n" + "="*60)
    print("PROMPT:", prompt)
    print("-"*60)
    print("BASE MODEL:")
    print(generate_text_with_model(model, prompt))
    print("\nPRUNED (last 2 removed):")
    print(generate_text_with_model(pruned_model_last2, prompt))

In [ ]:
mini_benchmark = [
    {"category": "qa", "prompt": "What is the capital of Germany?"},
    {"category": "math", "prompt": "What is 19 * 7?"},
    {"category": "code", "prompt": "Write a Python function to reverse a string."},
    {"category": "reasoning", "prompt": "If all roses are flowers and some flowers fade quickly, can we conclude all roses fade quickly? Why?"},
    {"category": "instruction", "prompt": "Give me 3 bullet points on why exercise is important."},
]

In [ ]:
print("=== MODEL FLOW ===")
print("Token IDs")
print("   ↓")
print("Embedding layer: model.model.embed_tokens")
print("   ↓")
for i in range(len(model.model.layers)):
    print(f"Transformer layer {i}: model.model.layers[{i}]")
print("   ↓")
print("Final norm: model.model.norm")
print("   ↓")
print("LM head: model.lm_head")
print("   ↓")
print("Next-token prediction")

In [ ]:
def test_model(current_model, label="Model"):
    prompts = [
        "What is 2 + 2?",                           # Math
        "Write a 1-line python code to print hello", # Code
        "Explain the moon in one sentence."         # Coherence
    ]
    print(f"\n--- Testing {label} ---")
    for p in prompts:
        inputs = tokenizer(p, return_tensors="pt").to(current_model.device)
        out = current_model.generate(**inputs, max_new_tokens=512, pad_token_id=tokenizer.eos_token_id)
        print(f"P: {p}\nA: {tokenizer.decode(out[0], skip_special_tokens=True)}\n")

# Run Phase 2 (Original)
test_model(model, "Original Qwen")

In [ ]:
import copy

# Create a copy so we don't destroy the original model
pruned_model = copy.deepcopy(model)

# Let's try removing layers from the middle (e.g., layers 10, 11, 12)
# IMPORTANT: Delete in reverse order or use slicing to avoid index shifts!
layers_to_remove = [10, 11, 12]

# Method: Re-assigning the layers list without the target indices
new_layers = torch.nn.ModuleList([
    layer for i, layer in enumerate(pruned_model.model.layers)
    if i not in layers_to_remove
])

pruned_model.model.layers = new_layers

print(f"New layer count: {len(pruned_model.model.layers)}")

In [ ]:
# Run Phase 5
test_model(pruned_model, "Pruned Qwen (Middle Layers Removed)")